# Model Comparison — DIC

In [15]:
import pandas as pd
import glob
import os

dic_files = sorted(glob.glob('../../data/models/*/dic.csv'))

rows = []
for f in dic_files:
    model = os.path.basename(os.path.dirname(f))
    df = pd.read_csv(f)
    df.insert(0, 'Model', model)
    rows.append(df)

df_dic = pd.concat(rows, ignore_index=True)
df_dic = df_dic.sort_values('DIC').reset_index(drop=True)
df_dic = df_dic.round(2)
df_dic.to_excel('../../data/models/dic_comparison.xlsx', index=False)
df_dic

,Model,deviance,penalty,DIC
0,v7,-4174.50,46.74,-4127.76
1,v6,2275.13,40.53,2315.66
2,v3,2277.94,46.76,2324.70
3,v5,2277.99,46.78,2324.78
4,v4,2278.04,46.75,2324.79
5,v9,2278.32,58.97,2337.30
6,v8,2279.01,58.94,2337.95
7,v2,2318.59,31.82,2350.41
8,v2a,2341.27,28.75,2370.03
9,v2b,2629.67,28.66,2658.33


# Convergence Diagnostics — Gelman-Rubin

In [16]:
import numpy as np

RHAT_THRESHOLD = 1.1

def gelman_rubin(chains):
    """R-hat for a single parameter. chains: list of 1-D arrays."""
    m = len(chains)
    n = min(len(c) for c in chains)
    chains = [c[:n] for c in chains]
    chain_means = np.array([c.mean() for c in chains])
    grand_mean = chain_means.mean()
    B = n * np.var(chain_means, ddof=1)
    W = np.mean([np.var(c, ddof=1) for c in chains])
    var_hat = ((n - 1) / n) * W + (1 / n) * B
    return np.sqrt(var_hat / W) if W > 0 else np.nan

sample_files = sorted(glob.glob('../../data/models/*/raw_samples.csv'))

summary_rows = []
for f in sample_files:
    model = os.path.basename(os.path.dirname(f))
    df = pd.read_csv(f)
    if 'chain' not in df.columns:
        summary_rows.append({
            'Model': model, 'n_params': len(df.columns),
            'max_rhat': np.nan, 'mean_rhat': np.nan,
            'n_flagged': np.nan, 'note': 'no chain column — re-knit'
        })
        continue

    chain_ids = sorted(df['chain'].unique())
    param_cols = [c for c in df.columns if c != 'chain']
    rhats = {}
    for p in param_cols:
        chains = [df.loc[df['chain'] == ch, p].values for ch in chain_ids]
        rhats[p] = gelman_rubin(chains)

    flagged = {p: r for p, r in rhats.items() if r > RHAT_THRESHOLD}
    summary_rows.append({
        'Model': model,
        'n_params': len(param_cols),
        'max_rhat': round(max(rhats.values()), 4),
        'mean_rhat': round(np.mean(list(rhats.values())), 4),
        'n_flagged': len(flagged),
        'note': ', '.join(f'{p} ({r:.3f})' for p, r in
                          sorted(flagged.items(), key=lambda x: -x[1])[:5])
               if flagged else 'all converged'
    })

df_gelman = pd.DataFrame(summary_rows).sort_values('Model').reset_index(drop=True)
df_gelman.to_excel('../../data/models/gelman_rubin_summary.xlsx', index=False)
df_gelman

,Model,n_params,max_rhat,mean_rhat,n_flagged,note
0,v0a,13,1.0000,1.0000,0,all converged
1,v0b,14,1.0001,1.0000,0,all converged
2,v1,25,1.0001,1.0000,0,all converged
3,v2,31,1.0002,1.0000,0,all converged
4,v2a,28,1.0000,1.0000,0,all converged
5,v2b,28,1.0001,1.0000,0,all converged
6,v3,46,1.0001,1.0000,0,all converged
7,v4,48,1.0001,1.0000,0,all converged
8,v5,55,1.0006,1.0001,0,all converged
9,v6,52,1.0007,1.0001,0,all converged
